In [ ]:
import os
import sys
import glob
import cv2
import numpy as np
import pandas as pd
import shutil
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')
from torch.utils.data import Dataset, DataLoader

REPO_ROOT = "d:/New folder/Non-Invasive/rPPG"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from neural_methods.model.RhythmFormer import RhythmFormer
from neural_methods.loss.RythmFormerLossComputer import RhythmFormer_Loss

In [ ]:
# ----- paths -----
RAW_DATA_PATH       = os.path.join(REPO_ROOT, "data/Headmotion")
PREPROCESSED_PATH   = os.path.join(REPO_ROOT, "preprocessed_data/Headmotion/groupG")
OUTPUT_MODEL_PATH   = os.path.join(REPO_ROOT, "final_model_release/GroupG_RhythmFormer.pth")

# ----- params -----
VIDEO_FPS    = 30
CHUNK_LENGTH = 160
IMG_H, IMG_W = 128, 128
BATCH_SIZE   = 4
EPOCHS       = 30
LR           = 3e-4

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

os.makedirs(PREPROCESSED_PATH, exist_ok=True)
os.makedirs(os.path.dirname(OUTPUT_MODEL_PATH), exist_ok=True)

In [ ]:
def read_video_frames(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret: break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return np.stack(frames, axis=0)

def read_ppg_synced(session_path, num_frames):
    frame_df = pd.read_csv(os.path.join(session_path, "frame_timestamps.csv"))
    min_len = min(len(frame_df), num_frames)
    frame_t = frame_df["timestamp"].values[:min_len]
    ppg_df = pd.read_csv(os.path.join(session_path, "ppg.csv"))
    ppg_t = ppg_df["Timestamp"].values
    ppg_val = ppg_df["PPG"].values
    frame_t_clipped = np.clip(frame_t, ppg_t[0], ppg_t[-1])
    ppg_resampled = np.interp(frame_t_clipped, ppg_t, ppg_val)
    return ppg_resampled.astype(np.float32)

def standardized_data(data):
    data = data.astype(np.float32)
    m = np.mean(data)
    s = np.std(data)
    if s > 0: data = (data - m) / s
    else: data = np.zeros_like(data)
    data = np.where(np.isnan(data), np.zeros_like(data), data)
    return data

def standardized_label(label):
    label = label.astype(np.float64)
    m = np.mean(label)
    s = np.std(label)
    if s > 0: label = (label - m) / s
    else: label = np.zeros_like(label)
    return label.astype(np.float32)

def crop_face_resize(frames, out_h, out_w, large_box_coef=1.5):
    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector  = cv2.CascadeClassifier(xml_path)
    frame0 = frames[0]
    if frame0.dtype != np.uint8:
        frame0 = np.clip(frame0, 0, 255).astype(np.uint8)
    gray = cv2.cvtColor(frame0, cv2.COLOR_RGB2GRAY)
    faces = detector.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    H, W = frames.shape[1], frames.shape[2]
    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])
        x  = max(0, int(x  - (large_box_coef - 1.0) / 2.0 * fw))
        y  = max(0, int(y  - (large_box_coef - 1.0) / 2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H
    C = frames.shape[3]
    resized = np.zeros((len(frames), out_h, out_w, C), dtype=np.float32)
    for i, frame in enumerate(frames):
        crop = frame[y : y + fh, x : x + fw]
        if crop.size == 0:
            crop = frame
        resized[i] = cv2.resize(crop.astype(np.float32), (out_w, out_h), interpolation=cv2.INTER_AREA)
    return resized

In [ ]:
all_dirs = sorted([d for d in glob.glob(os.path.join(RAW_DATA_PATH, "*")) if os.path.isdir(d) and os.path.basename(d) != "videos"])
subjects = []
for subj_dir in all_dirs:
    subj_id = os.path.basename(subj_dir)
    subj_key = subj_id.replace("_", "")
    video_files = glob.glob(os.path.join(RAW_DATA_PATH, "videos", f"{subj_id}.mkv"))
    if not video_files: continue
    subjects.append({"subj_id": subj_id, "subj_key": subj_key, "video_path": video_files[0], "session_path": subj_dir})

print(f"Total subjects: {len(subjects)}")

# Preprocessing
if os.path.exists(PREPROCESSED_PATH):
    shutil.rmtree(PREPROCESSED_PATH)
os.makedirs(PREPROCESSED_PATH)

all_input_files = []
for subj in subjects:
    subj_key = subj["subj_key"]
    frames = read_video_frames(subj["video_path"])
    T = frames.shape[0]
    ppg_signal = read_ppg_synced(subj["session_path"], T)
    frames_cropped = crop_face_resize(frames, IMG_H, IMG_W)
    data_normalized = standardized_data(frames_cropped)
    label_normalized = standardized_label(ppg_signal)
    
    clip_num = T // CHUNK_LENGTH
    data_clips  = np.array([data_normalized[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH]  for i in range(clip_num)])
    label_clips = np.array([label_normalized[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH] for i in range(clip_num)])
    
    subj_dir = os.path.join(PREPROCESSED_PATH, subj_key)
    os.makedirs(subj_dir, exist_ok=True)
    for chunk_idx in range(clip_num):
        input_path = os.path.join(subj_dir, f"{subj_key}_input{chunk_idx}.npy")
        label_path = os.path.join(subj_dir, f"{subj_key}_label{chunk_idx}.npy")
        np.save(input_path, data_clips[chunk_idx])
        np.save(label_path, label_clips[chunk_idx])
        all_input_files.append(input_path)

print(f"Total clips saved: {len(all_input_files)}")

In [ ]:
class RhythmFormerDataset(Dataset):
    def __init__(self, input_files):
        self.inputs = sorted(input_files)
        self.labels = [f.replace("input", "label") for f in self.inputs]
    def __len__(self):
        return len(self.inputs)
    def __getitem__(self, index):
        data  = np.float32(np.load(self.inputs[index]))
        label = np.float32(np.load(self.labels[index]))
        data = np.transpose(data, (0, 3, 1, 2))  # NDHWC -> NDCHW
        return data, label

dataset = RhythmFormerDataset(all_input_files)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
print(f"Dataset: {len(dataset)} clips")
print(f"DataLoader ready: {len(loader)} batches")

In [ ]:
model = RhythmFormer().to(DEVICE)
criterion = RhythmFormer_Loss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, epochs=EPOCHS, steps_per_epoch=max(1, len(loader))
)

diff_flag = 0  # Since LABEL_TYPE = "Standardized"


In [ ]:
for epoch in range(EPOCHS):
    print(f"\n====Training Epoch: {epoch}====")
    model.train()
    running_loss = 0.0
    
    tbar = tqdm(loader, ncols=80)
    for idx, batch in enumerate(tbar):
        tbar.set_description(f"Train epoch {epoch}")
        data, labels = batch[0].float(), batch[1].float()
        N, D, C, H, W = data.shape
        
        data = data.to(DEVICE)
        labels = labels.to(DEVICE)
        
        optimizer.zero_grad()
        pred_ppg = model(data)
        # normalize predictions
        pred_ppg = (pred_ppg - torch.mean(pred_ppg, dim=-1, keepdim=True)) / torch.std(pred_ppg, dim=-1, keepdim=True)
        
        loss = 0.0
        for ib in range(N):
            loss = loss + criterion(pred_ppg[ib], labels[ib], epoch, VIDEO_FPS, diff_flag)
        loss = loss / N
        loss.backward()
        
        optimizer.step()
        scheduler.step()
        
        running_loss += loss.item()
        tbar.set_postfix(loss=loss.item())
        
    print(f"Epoch {epoch} finished. Average Loss: {running_loss / max(1, len(loader)):.4f}")

torch.save(model.state_dict(), OUTPUT_MODEL_PATH)
print(f"\nTraining complete! Model saved to {OUTPUT_MODEL_PATH}")